# T2 — Extracción multifuente: SQL + API + privacidad

**Caso:** DataRetail LATAM necesita construir un dataset analítico a partir de una base SQLite interna y una API pública externa.

**Resultado esperado:** un archivo `dataset_final.csv` que integre ventas, clientes e información externa del país, eliminando identificadores personales directos.

**Tiempo sugerido:** 60 minutos.


## Objetivos

Al finalizar el taller deberías poder:

1. Conectarte a una base de datos SQLite desde Python.
2. Extraer datos mediante una consulta SQL.
3. Consultar una API usando `requests`.
4. Convertir una respuesta JSON en un `DataFrame`.
5. Integrar dos fuentes de datos con `merge`.
6. Eliminar o pseudonimizar identificadores personales.


In [58]:
import sqlite3
from pathlib import Path

import pandas as pd
import requests

DATA_DIR = Path('/content')
DB_PATH = DATA_DIR / 'ventas.db'

DB_PATH.exists(), DB_PATH

(True, PosixPath('/content/ventas.db'))

## 1. Conexión a la base de datos

Primero abriremos una conexión a `ventas.db`.


In [59]:
conn = sqlite3.connect(DB_PATH)
conn

### 1.1 ¿Qué tablas existen?

Completa la consulta para listar las tablas de la base.


In [60]:
query_tablas = """
SELECT name
FROM sqlite_master
WHERE type = 'table';
"""

pd.read_sql_query(query_tablas, conn)

,name
0,clientes
1,ventas


## 2. Primera exploración

Consulta algunas filas de cada tabla.


In [61]:
# TODO: consulta las primeras 5 filas de la tabla clientes
query_clientes = """
SELECT *
FROM clientes
LIMIT 5;
"""

pd.read_sql_query(query_clientes, conn)

,cliente_id,nombre,email,pais_iso2,segmento,edad,fecha_alta
0,1,Renata Morales,renata.morales1@dataretail.test,AR,Oro,30,2024-08-10
1,2,Vicente Pérez,vicente.perez2@dataretail.test,CL,Premium,20,2025-04-14
2,3,Diego Contreras,diego.contreras3@dataretail.test,CL,Oro,19,2024-09-22
3,4,Diego Rojas,diego.rojas4@dataretail.test,AR,Bronce,64,2025-03-22
4,5,Felipe Hernández,felipe.hernandez5@dataretail.test,CO,Bronce,38,2025-11-12


In [62]:
# TODO: consulta las primeras 5 filas de la tabla ventas
query_ventas = """
SELECT *
FROM ventas
LIMIT 5;
"""

pd.read_sql_query(query_ventas, conn)

,venta_id,cliente_id,fecha,monto,canal,categoria,cantidad,descuento
0,1,106,2026-02-06,82513.0,Tienda,Hogar,5,0.0
1,2,29,2026-04-07,27622.0,Marketplace,Libros,3,0.0
2,3,46,2026-05-13,12155.0,Marketplace,Alimentos,1,0.1
3,4,11,2026-02-24,41224.0,App,Alimentos,1,0.0
4,5,117,2026-07-15,81132.0,Marketplace,Alimentos,2,0.0


## 3. Extracción usando SQL

Queremos construir una tabla analítica que contenga información de ventas y de clientes.

Completa el `JOIN`.


In [63]:
query = """
SELECT
    v.venta_id,
    v.cliente_id,
    v.fecha,
    v.monto,
    v.canal,
    v.categoria,
    v.cantidad,
    v.descuento,
    c.nombre,
    c.email,
    c.pais_iso2,
    c.segmento,
    c.edad,
    c.fecha_alta
FROM ventas AS v
JOIN clientes AS c
    ON v.cliente_id = c.cliente_id
WHERE v.monto > 0
ORDER BY v.fecha;
"""

df_ventas = pd.read_sql_query(query, conn)
df_ventas.head()

,venta_id,cliente_id,fecha,monto,canal,categoria,cantidad,descuento,nombre,email,pais_iso2,segmento,edad,fecha_alta
0,146,61,2026-01-02,19179.0,Tienda,Libros,2,0.00,Camila Hernández,camila.hernandez61@dataretail.test,AR,Plata,23,2024-09-30
1,277,98,2026-01-03,60613.0,Marketplace,Tecnología,3,0.00,Valentina Hernández,valentina.hernandez98@dataretail.test,PE,Oro,41,2024-05-13
2,302,57,2026-01-03,46529.0,Web,Belleza,5,0.00,Antonia González,antonia.gonzalez57@dataretail.test,AR,Oro,48,2025-03-24
3,360,78,2026-01-03,33725.0,Tienda,Tecnología,1,0.15,Pedro Pérez,pedro.perez78@dataretail.test,CO,Bronce,21,2024-10-16
4,49,40,2026-01-05,56520.0,Web,Tecnología,2,0.00,Isidora Muñoz,isidora.muñoz40@dataretail.test,CO,Bronce,44,2025-11-03


Cuando la consulta funcione, revisa tamaño, columnas y países presentes.


In [64]:
# TODO: descomenta cuando exista df_ventas
df_ventas.shape
df_ventas.info()
df_ventas['pais_iso2'].unique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 443 entries, 0 to 442
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   venta_id    443 non-null    int64  
 1   cliente_id  443 non-null    int64  
 2   fecha       443 non-null    object 
 3   monto       443 non-null    float64
 4   canal       443 non-null    object 
 5   categoria   443 non-null    object 
 6   cantidad    443 non-null    int64  
 7   descuento   443 non-null    float64
 8   nombre      443 non-null    object 
 9   email       443 non-null    object 
 10  pais_iso2   443 non-null    object 
 11  segmento    443 non-null    object 
 12  edad        443 non-null    int64  
 13  fecha_alta  443 non-null    object 
dtypes: float64(2), int64(4), object(8)
memory usage: 48.6+ KB


array(['AR', 'PE', 'CO', 'BR', 'CL', 'MX'], dtype=object)

## 4. Consulta a una API

Usaremos la API del Banco Mundial para consultar información externa de países.

Primero prueba con Chile (`CL`).


In [65]:
codigo = 'CL'
url = f'https://api.worldbank.org/v2/country/{codigo}'
params = {'format': 'json'}

try:
    response = requests.get(url, params=params, timeout=10)
    print(response.status_code, response.url)
except Exception as error:
    print('No se pudo consultar la API. Más abajo hay una opción de respaldo local.')
    print(error)

200 https://api.worldbank.org/v2/country/CL?format=json


In [66]:
# Ejecuta esta celda solo si la consulta anterior a la API funcionó.
# Si no funcionó, avanza hasta la opción de emergencia.

datos = response.json()
datos

[{'page': 1, 'pages': 1, 'per_page': '50', 'total': 1},
 [{'id': 'CHL',
   'iso2Code': 'CL',
   'name': 'Chile',
   'region': {'id': 'LCN',
    'iso2code': 'ZJ',
    'value': 'Latin America & Caribbean '},
   'adminregion': {'id': '', 'iso2code': '', 'value': ''},
   'incomeLevel': {'id': 'HIC', 'iso2code': 'XD', 'value': 'High income'},
   'lendingType': {'id': 'IBD', 'iso2code': 'XF', 'value': 'IBRD'},
   'capitalCity': 'Santiago',
   'longitude': '-70.6475',
   'latitude': '-33.475'}]]

La respuesta tiene una estructura de lista. La información principal del país está en `datos[1][0]`.


In [67]:
pais = datos[1][0]

# TODO: inspecciona las llaves disponibles
pais.keys()

dict_keys(['id', 'iso2Code', 'name', 'region', 'adminregion', 'incomeLevel', 'lendingType', 'capitalCity', 'longitude', 'latitude'])

In [68]:
# TODO: extrae nombre, capital, región y nivel de ingreso
pais['name']
pais['capitalCity']
pais['region']['value']
pais['incomeLevel']['value']

'High income'

## 5. Función de extracción desde API

Completa una función que reciba un código ISO2 y devuelva un diccionario con información del país.


In [69]:
# Ejecuta esta celda solo si la consulta anterior a la API funcionó.
# Si no funcionó, avanza hasta la opción de emergencia.

datos = response.json()
datos
def obtener_info_pais(codigo):
    url = f'https://api.worldbank.org/v2/country/{codigo}'
    try:
        r = requests.get(url, params={"format": "json"}, timeout=10)
        r.raise_for_status()
        payload = r.json()
        pais = payload[1][0]
        return {
            "id_pais": pais["id"],
            "pais_iso2": codigo,
            "name": pais["name"],
            "region": pais["region"]["value"],
            "nivel_ingreso": pais["incomeLevel"]["value"],
            "capital": pais["capitalCity"]
        }
    except Exception as error:
        print(f"[Aviso] {codigo}: usando respaldo ({type(error).__name__})")
        return requests.get(codigo, {
            "pais_iso2": codigo,
            "pais": None,
            "capital": None,
            "region": None,
            "nivel_ingreso": None
        })

In [70]:
# TODO: probar función
obtener_info_pais('CL')

{'id_pais': 'CHL',
 'pais_iso2': 'CL',
 'name': 'Chile',
 'region': 'Latin America & Caribbean ',
 'nivel_ingreso': 'High income',
 'capital': 'Santiago'}

## 6. Obtener información para todos los países

Primero identifica los países presentes en `df_ventas`.


In [71]:
# TODO: países únicos de df_ventas
paises = df_ventas['pais_iso2'].unique()
paises

array(['AR', 'PE', 'CO', 'BR', 'CL', 'MX'], dtype=object)

In [72]:
# TODO: construir df_paises usando un loop
info_paises = []
for codigo in paises:
  info_paises.append(obtener_info_pais(codigo))

df_paises = pd.DataFrame(info_paises)
df_paises

,id_pais,pais_iso2,name,region,nivel_ingreso,capital
0,ARG,AR,Argentina,Latin America & Caribbean,Upper middle income,Buenos Aires
1,PER,PE,Peru,Latin America & Caribbean,Upper middle income,Lima
2,COL,CO,Colombia,Latin America & Caribbean,Upper middle income,Bogota
3,BRA,BR,Brazil,Latin America & Caribbean,Upper middle income,Brasilia
4,CHL,CL,Chile,Latin America & Caribbean,High income,Santiago
5,MEX,MX,Mexico,Latin America & Caribbean,Upper middle income,Mexico City


### Opción de emergencia si no hay internet

Si la API no responde durante la clase, ejecuta esta celda y construye `df_paises` desde el diccionario de respaldo.


In [73]:
# Diccionario de respaldo por si la API no responde durante la clase.
PaisesFallback = {
    'CL': {'pais_iso2':'CL', 'pais':'Chile', 'region':'Latin America & Caribbean', 'nivel_ingreso':'High income', 'capital':'Santiago'},
    'AR': {'pais_iso2':'AR', 'pais':'Argentina', 'region':'Latin America & Caribbean', 'nivel_ingreso':'Upper middle income', 'capital':'Buenos Aires'},
    'PE': {'pais_iso2':'PE', 'pais':'Peru', 'region':'Latin America & Caribbean', 'nivel_ingreso':'Upper middle income', 'capital':'Lima'},
    'CO': {'pais_iso2':'CO', 'pais':'Colombia', 'region':'Latin America & Caribbean', 'nivel_ingreso':'Upper middle income', 'capital':'Bogota'},
    'MX': {'pais_iso2':'MX', 'pais':'Mexico', 'region':'Latin America & Caribbean', 'nivel_ingreso':'Upper middle income', 'capital':'Mexico City'},
    'BR': {'pais_iso2':'BR', 'pais':'Brazil', 'region':'Latin America & Caribbean', 'nivel_ingreso':'Upper middle income', 'capital':'Brasilia'},
}

# df_paises = pd.DataFrame([PaisesFallback[c] for c in paises])
# df_paises

## 7. Integración de fuentes

Une `df_ventas` y `df_paises`.

Pregunta: ¿qué columna comparten ambas tablas?


In [74]:
# TODO: integrar por pais_iso2
df_integrado = df_ventas.merge(
    df_paises,
    on='pais_iso2',
    how='left'
)

df_integrado.head()

,venta_id,cliente_id,fecha,monto,canal,categoria,cantidad,descuento,nombre,email,pais_iso2,segmento,edad,fecha_alta,id_pais,name,region,nivel_ingreso,capital
0,146,61,2026-01-02,19179.0,Tienda,Libros,2,0.00,Camila Hernández,camila.hernandez61@dataretail.test,AR,Plata,23,2024-09-30,ARG,Argentina,Latin America & Caribbean,Upper middle income,Buenos Aires
1,277,98,2026-01-03,60613.0,Marketplace,Tecnología,3,0.00,Valentina Hernández,valentina.hernandez98@dataretail.test,PE,Oro,41,2024-05-13,PER,Peru,Latin America & Caribbean,Upper middle income,Lima
2,302,57,2026-01-03,46529.0,Web,Belleza,5,0.00,Antonia González,antonia.gonzalez57@dataretail.test,AR,Oro,48,2025-03-24,ARG,Argentina,Latin America & Caribbean,Upper middle income,Buenos Aires
3,360,78,2026-01-03,33725.0,Tienda,Tecnología,1,0.15,Pedro Pérez,pedro.perez78@dataretail.test,CO,Bronce,21,2024-10-16,COL,Colombia,Latin America & Caribbean,Upper middle income,Bogota
4,49,40,2026-01-05,56520.0,Web,Tecnología,2,0.00,Isidora Muñoz,isidora.muñoz40@dataretail.test,CO,Bronce,44,2025-11-03,COL,Colombia,Latin America & Caribbean,Upper middle income,Bogota


In [82]:
# TODO: revisar valores faltantes después del merge
df_integrado.isna().sum()

,0
venta_id,0
cliente_id,0
fecha,0
monto,0
canal,0
categoria,0
cantidad,0
descuento,0
nombre,0
email,0


In [83]:
df_ventas.shape

(443, 14)

In [84]:
df_integrado.shape

(443, 20)

In [85]:
df_integrado.columns

Index(['venta_id', 'cliente_id', 'fecha', 'monto', 'canal', 'categoria',
       'cantidad', 'descuento', 'nombre', 'email', 'pais_iso2', 'segmento',
       'edad', 'fecha_alta', 'id_pais', 'name', 'region', 'nivel_ingreso',
       'capital', 'cliente_anon'],
      dtype='object')

## 8. Privacidad: desidentificación

El dataset contiene identificadores directos como `nombre` y `email`. Para un análisis de ventas, estos campos no son necesarios.

Además, podemos reemplazar `cliente_id` por un identificador pseudónimo.


In [76]:
import hashlib

SECRETO = 'CD2026_TALLER'

def pseudonimizar(valor):
    texto = f'{SECRETO}|{valor}'
    return hashlib.sha256(texto.encode()).hexdigest()[:12]

In [77]:
# TODO: crear cliente_anon a partir de cliente_id
df_integrado['cliente_anon'] = df_integrado['cliente_id'].apply(pseudonimizar)
df_integrado[['cliente_id', 'cliente_anon']].head()

,cliente_id,cliente_anon
0,61,f2b1777ae260
1,98,ea6c1fef57d4
2,57,85e63ad45ddd
3,78,fd72b53db7a1
4,40,424e1423e582


In [78]:
# TODO: eliminar identificadores directos
columnas_eliminar = ['cliente_id', 'nombre', 'email']
df_final = df_integrado.drop(columns=columnas_eliminar)
df_final.head()

,venta_id,fecha,monto,canal,categoria,cantidad,descuento,pais_iso2,segmento,edad,fecha_alta,id_pais,name,region,nivel_ingreso,capital,cliente_anon
0,146,2026-01-02,19179.0,Tienda,Libros,2,0.00,AR,Plata,23,2024-09-30,ARG,Argentina,Latin America & Caribbean,Upper middle income,Buenos Aires,f2b1777ae260
1,277,2026-01-03,60613.0,Marketplace,Tecnología,3,0.00,PE,Oro,41,2024-05-13,PER,Peru,Latin America & Caribbean,Upper middle income,Lima,ea6c1fef57d4
2,302,2026-01-03,46529.0,Web,Belleza,5,0.00,AR,Oro,48,2025-03-24,ARG,Argentina,Latin America & Caribbean,Upper middle income,Buenos Aires,85e63ad45ddd
3,360,2026-01-03,33725.0,Tienda,Tecnología,1,0.15,CO,Bronce,21,2024-10-16,COL,Colombia,Latin America & Caribbean,Upper middle income,Bogota,fd72b53db7a1
4,49,2026-01-05,56520.0,Web,Tecnología,2,0.00,CO,Bronce,44,2025-11-03,COL,Colombia,Latin America & Caribbean,Upper middle income,Bogota,424e1423e582


## 9. Guardar dataset final

Guarda el resultado en `data/dataset_final.csv`.


In [79]:
# TODO: guardar archivo final
salida = DATA_DIR / 'dataset_final.csv'
df_final.to_csv(salida, index=False)
salida

PosixPath('/content/dataset_final.csv')

## 10. Preguntas de cierre

Responde brevemente:

1. ¿Qué información fue obtenida mediante SQL?
*   Que contenian las tablas de ventas y de clientes respectivamente
*   El join entre las dos tablas de datos
2. ¿Qué información fue obtenida mediante API?
*   Los datos de paises externos que son entre codigos de verificacion, region, capitales, coordenadas de localizacion, niveles de ingreso
3. ¿Qué variable permitió integrar ambas fuentes?
*   El codigo pais_iso2
4. ¿Por qué se eliminaron `nombre` y `email`?
*   Para mantener la integridad de datos personales de los clientes, evitando asi posibles conflictos de responsabilidad y etica de trabajo
5. ¿Qué ocurriría con el pipeline si la API deja de responder?
*   Dejaria de Funcionar respecto al uso de datos directo desde la API, por eso se recomienda en una de las casillas realizar a mano el guardado de datos para así que sea posible la utilizacion del pipeline sin conexion a internet
